# Per-annotation test analysis: V4 and V4oversample

本分析使用两个模型的 **epoch 18** checkpoint，在相同 testing split 上逐位医生评估。
每条记录保留自己的官方 mask、六项 semantic 评分及 malignancy 评分，不合并医生的 mask，也不平均评分。

1. **Semantic extractor**：原始 CT 与该医生的 mask 输入 MedicalNet，输出六个属性各自的五档概率。
   对每个 t=1,2,3,4,5，使用 `GT rating >= t` 为阳性，模型概率 `P(rating >= t)` 为连续 ROC score。
   t=1 全是阳性，AUC 为 N/A。
2. **Malignancy model**：官方 mask 与原始 HU 计算 18 项 radiomics，将其 baseline logit 与
   **同一位医生的六项官方 semantic 评分**产生的 semantic correction 合并，输出结节恶性概率。
   该医生的 malignancy 1–2 为阴性，4–5 为阳性；3 保留预测但排除于二分类 ROC。

模型参数、标准化参数和 risk bank 均冻结。选择 **visual** kernel，运行本 notebook 可查看已保存的完整结果，无需 GPU。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "back_prop/evaluate/per_annotation").is_dir()),
            Path("/home/users/yj255/code/imaging_feature"))
DIRECTORY = ROOT / "back_prop/evaluate/per_annotation"
RESULTS = DIRECTORY / "results"
audit = json.loads((RESULTS / "provenance.json").read_text())
summary = json.loads((RESULTS / "summary.json").read_text())
assert audit["status"] == "complete" and audit["full_test_set"]
assert summary["annotations"] == audit["expected_annotations"]
assert all(meta["bank_unchanged"] for meta in audit["models"].values())
annotations = pd.read_csv(RESULTS / "annotations.csv")
predictions = pd.read_csv(RESULTS / "predictions.csv")
print(f"CT scans: {audit['evaluated_cases']} | physical nodules: {audit['cohort']['physical_nodules']} | "
      f"reader annotations: {audit['expected_annotations']}")
display(pd.DataFrame([
    {"model": name, "epoch": meta["epoch"], "checkpoint": meta["checkpoint"],
     "risk bank models": meta["bank_models"], "test/training patient overlap": meta["test_patient_overlap"]}
    for name, meta in audit["models"].items()
]))
display(annotations["gt_malignancy"].value_counts().sort_index().rename("annotations").to_frame())

## Semantic threshold AUC

AUC 衡量连续预测分数对阳性/阴性的排序能力。下表和热图保留所有五个阈值。
每项 AUC 的阳性、阴性、有效样本数及无效原因可在 `semantic_threshold_auc.csv` 中查看。

In [ ]:
semantic_auc = pd.read_csv(RESULTS / "semantic_threshold_auc.csv")
semantic_table = semantic_auc.pivot(index=["model", "feature"], columns="threshold", values="auc")
display(semantic_table.round(4).fillna("N/A"))
display(Image(filename=str(RESULTS / "semantic_threshold_auc.png")))
display(FileLink("results/semantic_roc.pdf", result_html_prefix="All semantic ROC curves: "))

## Nodule malignancy ROC

预测概率沿用 checkpoint 的 evaluation 规则：对每个 risk-bank model 计算
`sigmoid(radiomics baseline logit + semantic correction)`，再对概率取均值。
评估单位是 annotation，因此同一个物理结节可能贡献多条来自不同医生的记录。

In [ ]:
malignancy_auc = pd.read_csv(RESULTS / "malignancy_auc.csv")
display(malignancy_auc[["model", "semantic_input", "auc", "n_evaluated", "n_positive", "n_negative",
                        "n_indeterminate", "n_invalid_input"]].round(4))
display(Image(filename=str(RESULTS / "malignancy_roc.png")))
display(FileLink("results/malignancy_roc.pdf", result_html_prefix="ROC PDF: "))

## 查看逐条预测

`predictions.csv` 保存两个模型的每条 annotation 结果，包括官方评分、预测的五档概率与期望评分、
18 项 radiomics 和最终 malignancy probability。相同 `annotation_key` 在两个模型之间一一对应。
空 mask 如存在，会保留记录并标明 `input_status`，不伪造预测。

In [ ]:
CASE_ID = None  # e.g. "LIDC-IDRI-0301__scan301"; None shows the first rows of the complete cohort
selected = predictions if CASE_ID is None else predictions[predictions.case_id == CASE_ID]
features = ["lobulation", "margin", "sphericity", "spiculation", "subtlety", "texture"]
columns = ["model", "annotation_key", "nodule_id", "input_status", "gt_malignancy", "malignancy_target",
           "radiomics_malignancy_probability", "malignancy_probability_official"]
columns += [name for feature in features for name in ("gt_"+feature, "pred_"+feature)]
display(selected[columns].head(30).round(4))

## 结果与复现

- `results/report.html`：完整图表报告。
- `results/annotations.csv`：独立医生标注构成的测试 cohort。
- `results/predictions.csv`：逐模型、逐 annotation 预测。
- `results/semantic_threshold_auc.csv`：两模型 × 六属性 × 五阈值的 AUC 与样本数。
- `results/malignancy_auc.csv`、`results/malignancy_roc_points.csv`：恶性 AUC 与 ROC 点。
- `results/provenance.json`：checkpoint SHA256、数据来源、预处理与冻结检查。

如需重新运行推理，在项目根目录执行：

```bash
sbatch back_prop/evaluate/per_annotation/run.sbatch
```

脚本会核对配置并复用已完成病例；更换 checkpoint 或评估口径时请使用新的 `--output` 目录。
只重新生成统计和图表，无需 GPU：

```bash
/home/users/yj255/.local/miniforge3/envs/pro6000/bin/python -m back_prop.evaluate.per_annotation.report back_prop/evaluate/per_annotation/results
```

这些是按 annotation 等权计算的点估计，同一患者/结节的多位医生记录存在相关性。
恶性标签来自 LIDC 医生评分；官方 semantic 输入用于单独评估 malignancy model。

In [ ]:
display(FileLink("results/report.html", result_html_prefix="Full report: "))
print("Results directory:", RESULTS)